In [2]:
# ============================================================
# Clone capstone repository into Colab runtime
# ============================================================

REPO_URL = "https://github.com/manuelarceaguirre/capstone-quantum.git"

%cd /content

# remove old runtime copy if it exists
!rm -rf capstone-quantum

# clone fresh copy
!git clone {REPO_URL}

# move into repo
%cd /content/capstone-quantum

# verify structure
!pwd
!ls

/content
Cloning into 'capstone-quantum'...
remote: Enumerating objects: 63, done.
remote: Counting objects: 100% (63/63), done.
remote: Compressing objects: 100% (47/47), done.
remote: Total 63 (delta 21), reused 52 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (63/63), 2.31 MiB | 32.41 MiB/s, done.
Resolving deltas: 100% (21/21), done.
/content/capstone-quantum
/content/capstone-quantum
data  docs  environment.yml  figures  notebooks  README.md  Week7.md


In [5]:
# ============================================================
# Cell 2: Benchmark-style pseudo-OOS split generator
# ============================================================

def generate_poos_splits(df, initial_train_size=120, test_size=1, step=1):
    """
    Generate benchmark-style pseudo-out-of-sample splits.

    Parameters
    ----------
    df : pd.DataFrame
        Time-indexed modeling dataframe.
    initial_train_size : int
        Number of observations in the rolling training window.
    test_size : int
        Number of observations in each test window.
    step : int
        Step size between forecast origins.

    Returns
    -------
    splits : list of dict
        Each split contains train/test integer locations and date ranges.
    """
    splits = []

    n = len(df)

    for train_start in range(0, n - initial_train_size - test_size + 1, step):
        train_end = train_start + initial_train_size
        test_start = train_end
        test_end = test_start + test_size

        split = {
            "train_idx": list(range(train_start, train_end)),
            "test_idx": list(range(test_start, test_end)),
            "train_start_date": df.index[train_start],
            "train_end_date": df.index[train_end - 1],
            "test_start_date": df.index[test_start],
            "test_end_date": df.index[test_end - 1],
        }

        splits.append(split)

    return splits


# ------------------------------------------------------------
# Generate splits on Track B directly for now
# ------------------------------------------------------------

poos_splits = generate_poos_splits(
    track_b_df,
    initial_train_size=120,
    test_size=1,
    step=1
)

print("Number of POOS splits:", len(poos_splits))

print("\nFirst split:")
print(poos_splits[0])

print("\nLast split:")
print(poos_splits[-1])

Number of POOS splits: 647

First split:
{'train_idx': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119], 'test_idx': [120], 'train_start_date': Timestamp('1962-04-01 00:00:00'), 'train_end_date': Timestamp('1972-03-01 00:00:00'), 'test_start_date': Timestamp('1972-04-01 00:00:00'), 'test_end_date': Timestamp('1972-04-01 00:00:00')}

Last split:
{'train_idx': [646, 647, 648, 649, 650, 651, 652, 653, 654, 655, 656, 657, 658, 659, 660, 661, 662, 663, 664, 665, 666, 667, 668, 669, 670, 671, 672, 673, 674, 675, 676, 677, 678, 679, 680, 681, 682, 683, 684, 6

In [6]:
# ============================================================
# Cell 3: Create 6-lag supervised learning matrix
# ============================================================

def create_lagged_features(df, target_col, n_lags=6):
    """
    Create supervised learning matrix using lags of all Track B variables.

    X at time t contains variables from:
        t-1, t-2, ..., t-6

    y at time t is:
        target_col at time t

    This follows a direct one-step-ahead forecasting setup.
    """
    lagged_parts = []

    for lag in range(1, n_lags + 1):
        lagged = df.shift(lag).add_suffix(f"_lag{lag}")
        lagged_parts.append(lagged)

    X = pd.concat(lagged_parts, axis=1)
    y = df[target_col].copy()

    modeling_df = pd.concat([X, y.rename("target")], axis=1).dropna()

    X = modeling_df.drop(columns=["target"])
    y = modeling_df["target"]

    return X, y, modeling_df


TARGET_COL = "UNRATE"
N_LAGS = 6

X_lagged, y_lagged, modeling_df = create_lagged_features(
    track_b_df,
    target_col=TARGET_COL,
    n_lags=N_LAGS
)

print("Target:", TARGET_COL)
print("Number of lags:", N_LAGS)

print("\nX shape:", X_lagged.shape)
print("y shape:", y_lagged.shape)
print("Modeling df shape:", modeling_df.shape)

print("\nDate range after lagging:")
print(modeling_df.index.min(), "->", modeling_df.index.max())

print("\nFirst X columns:")
print(X_lagged.columns[:15].tolist())

print("\nMissing values in modeling df:", modeling_df.isna().sum().sum())

display(modeling_df.head())

Target: UNRATE
Number of lags: 6

X shape: (761, 30)
y shape: (761,)
Modeling df shape: (761, 31)

Date range after lagging:
1962-10-01 00:00:00 -> 2026-02-01 00:00:00

First X columns:
['UNRATE_lag1', 'PERMIT_lag1', 'S&P 500_lag1', 'UMCSENTx_lag1', 'T10Y3M_level_lag1', 'UNRATE_lag2', 'PERMIT_lag2', 'S&P 500_lag2', 'UMCSENTx_lag2', 'T10Y3M_level_lag2', 'UNRATE_lag3', 'PERMIT_lag3', 'S&P 500_lag3', 'UMCSENTx_lag3', 'T10Y3M_level_lag3']

Missing values in modeling df: 0


,UNRATE_lag1,PERMIT_lag1,S&P 500_lag1,UMCSENTx_lag1,T10Y3M_level_lag1,UNRATE_lag2,PERMIT_lag2,S&P 500_lag2,UMCSENTx_lag2,T10Y3M_level_lag2,...,PERMIT_lag5,S&P 500_lag5,UMCSENTx_lag5,T10Y3M_level_lag5,UNRATE_lag6,PERMIT_lag6,S&P 500_lag6,UMCSENTx_lag6,T10Y3M_level_lag6,target
date,,,,,,,,,,,,,,,,,,,,,
1962-10-01,-0.1,7.109062,-0.008926,0.0,1.20,0.3,7.090077,0.026844,-3.8,1.16,...,7.040536,-0.077267,-4.5,1.18,0.0,7.118826,-0.032387,0.0,1.11,-0.2
1962-11-01,-0.2,7.074117,-0.032060,0.0,1.19,-0.1,7.109062,-0.008926,0.0,1.20,...,7.050989,-0.124253,0.0,1.18,-0.1,7.040536,-0.077267,-4.5,1.18,0.3
1962-12-01,0.3,7.119636,0.066628,3.4,1.09,-0.2,7.074117,-0.032060,0.0,1.19,...,7.080868,0.023802,0.0,1.09,0.0,7.050989,-0.124253,0.0,1.18,-0.2
1963-01-01,-0.2,7.119636,0.042393,0.0,0.99,0.3,7.119636,0.066628,3.4,1.09,...,7.090077,0.026844,-3.8,1.16,-0.1,7.080868,0.023802,0.0,1.09,0.2
1963-02-01,0.2,7.129298,0.037906,0.0,0.92,-0.2,7.119636,0.042393,0.0,0.99,...,7.109062,-0.008926,0.0,1.20,0.3,7.090077,0.026844,-3.8,1.16,0.2


In [7]:
# ============================================================
# Cell 4: LinearSVR pseudo-OOS forecasting loop
# ============================================================

# rebuild splits on modeling_df AFTER lagging
poos_splits = generate_poos_splits(
    modeling_df,
    initial_train_size=120,
    test_size=1,
    step=1
)

# ------------------------------------------------------------
# Model pipeline
# ------------------------------------------------------------

svr_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svr", LinearSVR(
        C=1.0,
        epsilon=0.1,
        max_iter=20000,
        random_state=42
    ))
])

# ------------------------------------------------------------
# Forecast loop
# ------------------------------------------------------------

predictions = []
actuals = []
forecast_dates = []

for split in poos_splits:

    train_idx = split["train_idx"]
    test_idx = split["test_idx"]

    # train/test split
    X_train = X_lagged.iloc[train_idx]
    y_train = y_lagged.iloc[train_idx]

    X_test = X_lagged.iloc[test_idx]
    y_test = y_lagged.iloc[test_idx]

    # fit model
    svr_pipeline.fit(X_train, y_train)

    # predict
    y_pred = svr_pipeline.predict(X_test)[0]

    predictions.append(y_pred)
    actuals.append(y_test.values[0])
    forecast_dates.append(y_test.index[0])

# ------------------------------------------------------------
# Results dataframe
# ------------------------------------------------------------

results_df = pd.DataFrame({
    "date": forecast_dates,
    "actual": actuals,
    "prediction": predictions
})

results_df["abs_error"] = np.abs(
    results_df["actual"] - results_df["prediction"]
)

results_df["sq_error"] = (
    results_df["actual"] - results_df["prediction"]
) ** 2

# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------

mae = results_df["abs_error"].mean()
rmse = np.sqrt(results_df["sq_error"].mean())

print("=" * 60)
print("LINEAR SVR RESULTS")
print("=" * 60)

print(f"Forecast observations: {len(results_df)}")
print(f"MAE:  {mae:.6f}")
print(f"RMSE: {rmse:.6f}")

display(results_df.head())

display(results_df.tail())

/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  w

LINEAR SVR RESULTS
Forecast observations: 641
MAE:  0.176370
RMSE: 0.501184


,date,actual,prediction,abs_error,sq_error
0,1972-10-01,0.1,0.055536,0.044464,0.001977
1,1972-11-01,-0.3,0.042245,0.342245,0.117131
2,1972-12-01,-0.1,-0.034641,0.065359,0.004272
3,1973-01-01,-0.3,-0.025781,0.274219,0.075196
4,1973-02-01,0.1,-0.028444,0.128444,0.016498


,date,actual,prediction,abs_error,sq_error
636,2025-10-01,0.0,-0.105612,0.105612,0.011154
637,2025-11-01,0.1,-0.254130,0.354130,0.125408
638,2025-12-01,-0.1,-0.179540,0.079540,0.006327
639,2026-01-01,-0.1,-0.117085,0.017085,0.000292
640,2026-02-01,0.1,0.045030,0.054970,0.003022


In [9]:
# ============================================================
# Cell 5: Benchmark-consistent MASE computations
# Matches benchmark notebook: lag-1 naive MAE on full target series
# ============================================================

y_true_all = np.array(actuals)
y_pred_all = np.array(predictions)

# model MAE
mae = np.mean(np.abs(y_true_all - y_pred_all))

# benchmark MASE denominator: lag-1 naive forecast on FULL target series
y_full = track_b_df[TARGET_COL].values
naive_mae = np.mean(np.abs(y_full[1:] - y_full[:-1]))

mase = mae / naive_mae

print("=" * 60)
print("LINEAR SVR METRICS — BENCHMARK-CONSISTENT")
print("=" * 60)
print(f"MAE:        {mae:.6f}")
print(f"RMSE:       {rmse:.6f}")
print(f"Naive MAE:  {naive_mae:.6f}")
print(f"MASE:       {mase:.6f}")

LINEAR SVR METRICS — BENCHMARK-CONSISTENT
MAE:        0.176370
RMSE:       0.501184
Naive MAE:  0.208747
MASE:       0.844900
